In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from fit_config import build_full_model, build_continuum_model

In [ ]:
cont_model, cont_template = build_continuum_model(0.1)
theta_best_cont = cont_model.theta

full_model, full_template = build_full_model(
    continuum_template=cont_template,
    theta_best_cont=theta_best_cont,
    cont_model=cont_model,
    redshift=0.1,
)

In [ ]:
agebins = full_model.params["agebins"]
dt = 10 ** agebins[:, 1] - 10 ** agebins[:, 0]  # in yr

In [ ]:
N = 1000
sfrs = np.zeros((N, len(dt)))
mtot = np.zeros(N)

In [ ]:
for k in range(N):
    u = np.random.uniform(0, 1, full_model.ndim)

    i_logmass = full_model.theta_labels().index("logmass")
    fixed_logmass = 10.0

    theta = full_model.prior_transform(u)
    theta[i_logmass] = fixed_logmass

    full_model.set_parameters(theta)

    masses = np.atleast_1d(full_model.params["mass"].copy())
    sfrs[k] = masses / dt
    mtot[k] = np.sum(masses)


ssfr = sfrs / mtot[:, None]

In [ ]:
t_plot = (10**agebins / 1e9).flatten()
ssfr_step = np.repeat(ssfr, 2, axis=1)
sfr_step = np.repeat(sfrs, 2, axis=1)

In [ ]:
q16, q50, q84 = np.percentile(np.log10(ssfr_step), [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(t_plot, q16, q84, color="C0", alpha=0.25, label="16-84%")
ax.plot(t_plot, q50, color="C0", lw=2, label="median")
# ax.set_xscale("log")
ax.set_xlabel("Lookback time [Gyr]")
ax.set_ylabel(r"$\log\,\mathrm{sSFR}(t)\ [\mathrm{yr}^{-1}]$")

ax.set_xlim(0, 0.350)

ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
q16, q50, q84 = np.percentile(np.log10(sfr_step), [16, 50, 84], axis=0)

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(t_plot, q16, q84, color="C0", alpha=0.25, label="16-84%")
ax.plot(t_plot, q50, color="C0", lw=2, label="median")
# ax.set_xscale("log")
ax.set_xlabel("Lookback time [Gyr]")
ax.set_ylabel(r"$\log\,\mathrm{SFR}(t)\ [M_\odot\,\mathrm{yr}^{-1}]$")

ax.set_xlim(0, 0.350)

ax.legend()
plt.tight_layout()
plt.show()